# GNN3: FPL + GNN Residual Surrogate (CLEAN COLAB ENTRY)

Use this notebook in Colab to avoid any stale imports or old cells.

Pipeline:
1. Install dependencies (OpenDSSDirect, PyTorch, PyG).
2. Clone / update `GNN-Sandia` and `cd` into it.
3. Import FPL+GNN helpers from `fpl_gnn/`.
4. Generate dataset and train the residual GNN.

**Important:** Always open this notebook from GitHub (File → Open notebook → GitHub) to ensure you get the latest version.

In [ ]:
# 1) Core Python deps + OpenDSSDirect
!pip install -q pandas matplotlib
!pip install -q git+https://github.com/dss-extensions/OpenDSSDirect.py.git@master

# 2) Torch + PyTorch Geometric (for Colab CUDA 12.1 runtime)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f https://data.pyg.org/whl/torch-2.2.0+cu121.html

In [ ]:
# 2) Colab setup: clone repo and force-sync to origin/main
import os

REPO_DIR = "/content/GNN-Sandia"
REPO_URL = "https://github.com/alitasavori/GNN-Sandia.git"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Force-sync to latest main to avoid stale files
!git fetch origin
!git reset --hard origin/main

print(f"Working directory: {os.getcwd()}")
!git rev-parse --short HEAD

In [ ]:
# 3) Imports (reload + sanity checks)
import importlib
import inspect
import torch
import numpy as np
import pandas as pd

import fpl_gnn.fpl_gnn_dataset as ds
import fpl_gnn.fpl_gnn_train as tr

importlib.reload(ds)
importlib.reload(tr)

from fpl_gnn.fpl_gnn_dataset import generate_fpl_residual_dataset
from fpl_gnn.fpl_gnn_train import train_fpl_gnn

print("generate_fpl_residual_dataset signature:\n ", inspect.signature(generate_fpl_residual_dataset))
print("train_fpl_gnn signature:\n ", inspect.signature(train_fpl_gnn))

# Hard fail early if Colab somehow imported an old version
assert "mu_P_load_kw" in str(inspect.signature(generate_fpl_residual_dataset)), "Stale dataset module: missing mu_P_load_kw"

In [ ]:
# 4) Generate FPL residual dataset (scenario draws from Normal distributions)
# Means are the inputs; sigma_shared is the std-dev used for all 4 draws.
df_s, df_n, node_csv = generate_fpl_residual_dataset(
    n_scenarios=500,
    mu_P_load_kw=1415.2,
    mu_Q_load_kvar=835.2,
    mu_P_der_kw=1000.0,
    mu_Q_der_kvar=0.0,
    sigma_shared=50.0,
)
print("Dataset generation complete.")

In [ ]:
# 5) Train FPL+MLP residual model
model_fpl_mlp = train_fpl_gnn(
    hidden_dims=(128, 128, 64),
    batch_size=64,
    epochs=50,
    test_frac=0.2,
)

print("Training complete.")